In [ ]:
DDP-FSDP (Fully Sharded Data Parallel)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms, models
from torch.amp import GradScaler, autocast
import os

def train():
    # 1. 设置后端与分布式环境
    dist.init_process_group("nccl") #初始化分布式进程组
    local_rank = int(os.environ["LOCAL_RANK"]) #获取当前进程在本地的编号(0-7)
    torch.cuda.set_device(local_rank) #全局状态设置 在这个进程中，接下来的所有 CUDA 操作，默认都去操作 local_rank 这张卡
    device = torch.device(f"cuda:{local_rank}")

    # 2. 高效数据预处理
    transform = transforms.Compose([
        transforms.RandomResizedCrop(224), #随机裁剪并缩放到224x224
        #让模型学会识别目标物体在图片中不同位置、不同大小的情况，防止模型只盯着图中心看
        transforms.RandomHorizontalFlip(), #随机水平翻转，增加数据多样性
        transforms.ToTensor(), 
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    # 下载数据 (建议每个进程只在 rank 0 下载)
    # dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
    #上面的不好
    
    #------------------------------------------------------------
    # 1. 只有 Rank 0 执行下载
    if dist.get_rank() == 0:
        # 这一步会自动下载并解压，因为 download=True
        datasets.CIFAR100(root='./data', train=True, download=True)
        print("Rank 0: 数据集已准备就绪。")

    # 2. 【核心】进程屏障：直到所有进程都执行到了各自代码中的 dist.barrier() 才会继续往下走
    dist.barrier() #确保 Rank 0 下载完成后，其他进程再继续往下执行

    # 3. 现在所有进程都可以安全地加载数据集了
    dataset = datasets.CIFAR100(root='./data', train=True, download=False, transform=transform)
    #------------------------------------------------------------
    
    # 分布式采样
    sampler = DistributedSampler(dataset)
    loader = DataLoader(dataset, batch_size=128, sampler=sampler, num_workers=4, pin_memory=True)
    #这是单卡的 Batch Size。你的全局 Batch Size 实际上是 128 * 8 = 1024
    # 告诉 DataLoader：“不要用默认的顺序采样，请按照我上面分好的 DistributedSampler 逻辑来取数据
    # Pinned Memory 显存锁页内存
    
    # 3. 加载更现代的模型 (ResNet18)
    model = models.resnet18(num_classes=100).to(device)
    model = DDP(model, device_ids=[local_rank]) #将模型封装进 DDP容器

    # 4. 优化器与混合精度
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda') # 使用 AMP 减少显存占用.初始化梯度缩放器 (Gradient Scaler)，用于自动混合精度训练 (AMP)。

    # 5. 训练循环
    for epoch in range(10):
        sampler.set_epoch(epoch) #单卡训练中，直接用 DataLoader(shuffle=True)；DDP中在每一epoch开始前，调整随机种子，重新洗牌，让每张卡在新的 Epoch 拿到不同于上一轮的数据片段。
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            
            # 开启自动混合精度训练 性能优化
            with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                output = model(data)
                loss = nn.functional.cross_entropy(output, target)
            
            scaler.scale(loss).backward()
            #由于使用了 FP16，梯度可能会非常小以至于消失，scaler 对 loss 进行了放大，确保梯度可以被精确计算。
            scaler.step(optimizer)
            scaler.update()

        if dist.get_rank() == 0:
            #只有 Rank 0 进程负责打印日志，避免多个进程同时输出混乱的日志信息
            print(f"Epoch {epoch} completed. Loss: {loss.item():.4f}")

    dist.destroy_process_group() #训练完成后销毁分布式进程组，释放资源

if __name__ == "__main__":
    train()

# torchrun --nproc_per_node=8 run.py
# (number of processes进程数)

In [ ]:
# 模板
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

# 1. 初始化 (标准化)
local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
dist.init_process_group("nccl", device_id=torch.device(f"cuda:{local_rank}"))

# 2. 模型定义 (业务逻辑)
model = MyModel().to(local_rank)
model = DDP(model, device_ids=[local_rank])

# 3. 数据采样 (标准化)
sampler = DistributedSampler(dataset)
loader = DataLoader(dataset, sampler=sampler, ...)

# 4. 训练循环 (标准化 + 业务逻辑)
for epoch in range(10):
    sampler.set_epoch(epoch) # 必须有
    for data, target in loader:
        # ... 混合精度计算 ...
        # ... 梯度回传 ...

In [ ]:
# 观察gpu情况
nvtop

watch -n 0.5 nvidia-smi

